# CUDA 설치 확인

In [1]:
import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
print(f"현재 GPU 이름: {torch.cuda.get_device_name(0)}")

PyTorch 버전: 2.5.1+cu121
CUDA 사용 가능 여부: True
현재 GPU 이름: NVIDIA GeForce GTX 1070


# 라이브러리 임포트

In [2]:
import os
import json
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

# GTX 1070 사용을 위한 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")

현재 사용 중인 장치: cuda


# 정답 데이터 로드

In [16]:
# 상대경로 설정
label_dir = "./Train/morpheme/01/"
# label_dir = "./Test/labels/"
label_paths = glob.glob(os.path.join(label_dir, "*_morpheme.json"))

label_list = []

for path in label_paths[:500]:
    with open(path, 'r', encoding='utf-8') as f:
        content = json.load(f)
        
        # 1. 원본 영상 파일명 (morpheme.json에서 추출)
        full_name = content['metaData']['name'] # 예: NIA_SL_WORD0001_REAL01_D.mp4
        base_name = full_name.replace('.mp4', '') # 확장자 제거
        
        # 2. 정답 단어 및 시간 정보[cite: 1]
        word = content['data'][0]['attributes'][0]['name']
        start_t = content['data'][0]['start']
        end_t = content['data'][0]['end']
        
        label_list.append({
            'folder_name': base_name, # 키포인트 폴더 이름과 일치
            'word': word,
            'start': start_t,
            'end': end_t
        })

df = pd.DataFrame(label_list)
print(f"총 {len(df)}개의 정답지를 불러왔습니다.")
display(df.head()) # 데이터가 잘 들어왔는지 표로 확인

총 500개의 정답지를 불러왔습니다.


,folder_name,word,start,end
0,NIA_SL_WORD0001_REAL01_D,고민,1.743,3.103
1,NIA_SL_WORD0001_REAL01_F,고민,1.743,3.103
2,NIA_SL_WORD0001_REAL01_L,고민,1.743,3.103
3,NIA_SL_WORD0001_REAL01_R,고민,1.743,3.103
4,NIA_SL_WORD0001_REAL01_U,고민,1.743,3.103


# 데이터 가공 (키포인트 추출)

In [17]:
def get_sequence_data(folder_path, start_t, end_t, fps=30):
    # 폴더 내 모든 프레임 json 파일을 이름순으로 정렬
    json_files = sorted(glob.glob(os.path.join(folder_path, "*.json")))
    
    # 정답지에 적힌 시간대에 해당하는 프레임만 계산[cite: 1]
    start_idx = int(start_t * fps)
    end_idx = int(end_t * fps)
    
    # 해당 구간의 파일만 선택
    selected_files = json_files[start_idx : end_idx]
    
    sequence = []
    for file in selected_files:
        with open(file, 'r') as f:
            data = json.load(f)
            # 사용할 좌표 데이터 추출 (예: 132개 좌표)
            # 여기서는 구조만 잡기 위해 랜덤 값을 넣습니다. 실제 데이터 위치에 맞게 수정 필요.
            points = np.random.rand(132) 
            sequence.append(points)
            
    return np.array(sequence)

# 실제 데이터 로드 루프
X_list = []
y_list = []

# 단어 이름을 숫자로 바꾸는 사전 (예: "고민" -> 0)
word_to_idx = {word: i for i, word in enumerate(df['word'].unique())}

data_root = "./Train/01/"
# data_root = "./Test/datas/"

for idx, row in tqdm(df.iterrows(), total=len(df)):
    folder_path = os.path.join(data_root, row['folder_name'])
    
    if os.path.exists(folder_path):
        seq = get_sequence_data(folder_path, row['start'], row['end'])
        if len(seq) > 0:
            X_list.append(seq)
            y_list.append(word_to_idx[row['word']])

print(f"학습용 시퀀스 {len(X_list)}개 생성 완료!")

  0%|          | 0/500 [00:00<?, ?it/s]

학습용 시퀀스 500개 생성 완료!


# 모델 학습을 위한 데이터셋 및 데이터

In [18]:
from torch.nn.utils.rnn import pad_sequence

# 1. 시퀀스 길이 맞추기 (Padding)
# 영상마다 프레임 수가 다르므로, 가장 긴 것에 맞춰 0을 채워넣습니다.
X_tensors = [torch.FloatTensor(x) for x in X_list]
X_padded = pad_sequence(X_tensors, batch_first=True) # (전체데이터, 최대프레임, 132)

# 2. 정답 데이터 텐서 변환
y_tensor = torch.LongTensor(y_list)

# 3. 데이터셋 분리 (훈련 80%, 검증 20%)
dataset = TensorDataset(X_padded, y_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# 4. 데이터로더 설정 (GTX 1070 고려해서 배치 사이즈 16)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"학습 준비 완료! 배치 개수: {len(train_loader)}")

학습 준비 완료! 배치 개수: 50


# 모델 학습

In [20]:
# 모델 구조
class SignLanguageClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(SignLanguageClassifier, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: (Batch, Seq_len, Input_dim)
        out, _ = self.gru(x)
        # 마지막 프레임의 결과만 사용해서 분류
        out = self.fc(out[:, -1, :])
        return out

# 하이퍼파라미터 설정
input_size = 132 # 추출한 좌표 개수
hidden_size = 64
num_layers = 2
num_classes = len(word_to_idx)

model = SignLanguageClassifier(input_size, hidden_size, num_classes, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 실제 학습 루프
num_epochs = 100
pbar = tqdm(range(num_epochs), desc="전체 학습 진행도")
for epoch in pbar:
    model.train()
    train_loss = 0
    for batch_X, batch_y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    print(f"Epoch {epoch+1} Loss: {train_loss/len(train_loader):.4f}")

전체 학습 진행도:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1/100:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1 Loss: 4.6085


Epoch 2/100:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2 Loss: 4.4618


...

Epoch 98 Loss: 1.3532


Epoch 99/100:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 99 Loss: 1.3495


Epoch 100/100:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 100 Loss: 1.3403


# 모델 평가 및 결과 확인

In [22]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in val_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

print(f"검증 데이터 정확도: {100 * correct / total:.2f}%")

검증 데이터 정확도: 20.00%


In [23]:
# [셀 6-2] Top-5 정확도 확인 (정답이 상위 5개 후보 안에 들어있는지)
def get_top_n_accuracy(model, loader, n=5):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, pred = outputs.topk(n, 1, True, True) # 상위 n개 추출
            correct += pred.eq(batch_y.view(-1, 1).expand_as(pred)).sum().item()
            total += batch_y.size(0)
    return 100 * correct / total

top5_acc = get_top_n_accuracy(model, val_loader, n=5)
print(f"Top-5 정확도: {top5_acc:.2f}%")

Top-5 정확도: 78.00%


# 모델 저장

In [9]:
save_path = "./sign_model_v1.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'word_to_idx': word_to_idx
}, save_path)
print(f"모델이 {save_path}에 저장되었습니다.")

모델이 ./sign_model_v1.pth에 저장되었습니다.


# 모델 불러오기

In [10]:
# 1. 모델 구조 다시 정의 (저장할 때와 동일해야 함)
# 위에서 이미 정의했다면 생략 가능하지만, 새 파일에서 불러올 땐 필요합니다.
model_load = SignLanguageClassifier(input_size, hidden_size, num_classes, num_layers).to(device)

# 2. 저장된 파일 불러오기
checkpoint = torch.load("./sign_model_v1.pth", map_location=device)

# 3. 모델 가중치 복구
model_load.load_state_dict(checkpoint['model_state_dict'])

# 4. 단어 사전 복구
loaded_word_to_idx = checkpoint['word_to_idx']
# 반대로 숫자에서 단어를 찾는 사전도 만들어둡니다.
idx_to_word = {i: word for word, i in loaded_word_to_idx.items()}

model_load.eval() # 추론 모드로 전환 (Dropout 등이 비활성화됨)
print("모델 및 단어 사전 불러오기 완료!")

모델 및 단어 사전 불러오기 완료!


/tmp/ipykernel_942/1973582600.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./sign_model_v1.pth", map_location=device)


# 불러온 모델 테스트

In [11]:
# 검증 데이터셋에서 샘플 하나 꺼내오기
test_input, test_label = val_dataset[0] 
test_input = test_input.unsqueeze(0).to(device) # 배치를 위한 차원 추가 (1, Seq, 132)

# 모델 예측
with torch.no_grad():
    output = model_load(test_input)
    _, predicted_idx = torch.max(output, 1)

# 결과 출력
pred_word = idx_to_word[predicted_idx.item()]
real_word = idx_to_word[test_label.item()]

print(f"모델의 예측: {pred_word}")
print(f"실제 정답: {real_word}")

모델의 예측: 발가락
실제 정답: 발가락
